# Project 01 (basic) — An n-gram language model

**Module 08 — NLP 1**

You build a statistical **language model** on real text (*The Adventures of
Sherlock Holmes*): n-gram counts, **add-k smoothing**, **interpolation**,
evaluation by **perplexity** and **text generation** by sampling (script part 1).

You experience directly: why unsmoothed models fail, why higher n-grams have a
lower perplexity (until the data become too thin) and why mixing (interpolation)
works best.

## Setup
Only the standard library. The first cell **downloads the corpus** from Project
Gutenberg (about 600 KB) into `datasets/` and caches it (the file is not checked
in, via `.gitignore`). Select the kernel of the repository `.venv` and run the
cells in order; then solve tasks 1–3.

## Part A — Data and preprocessing (given)
Load the corpus, split it into sentences and tokens, split train/test, build the
vocabulary with `<unk>`/`<s>`/`</s>` and count the n-grams.

In [ ]:
# ---- Load the corpus (real text: "The Adventures of Sherlock Holmes") -----
import os, re, urllib.request, math, random
from collections import Counter, defaultdict

DATA_DIR = "datasets"
os.makedirs(DATA_DIR, exist_ok=True)
CORPUS = os.path.join(DATA_DIR, "sherlock.txt")
URL = "https://www.gutenberg.org/files/1661/1661-0.txt"

if not os.path.exists(CORPUS):
    print("Downloading the corpus from Project Gutenberg ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req).read().decode("utf-8", errors="ignore")
    # remove the Gutenberg boilerplate (licence etc.)
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    with open(CORPUS, "w", encoding="utf-8") as f:
        f.write(raw)
    print("saved:", CORPUS)

text = open(CORPUS, encoding="utf-8").read()
print(f"Corpus length: {len(text):,} characters")

In [ ]:
# ---- Tokenization and sentence segmentation --------------------------------
def sentences(text):
    """A very simple segmentation: split sentences at .!?, then into words."""
    text = text.replace("\n", " ")
    for raw in re.split(r"[.!?]+", text):
        toks = re.findall(r"[a-z']+", raw.lower())
        if toks:
            yield toks

sents = list(sentences(text))
random.seed(42); random.shuffle(sents)
split = int(0.9 * len(sents))
train_sents, test_sents = sents[:split], sents[split:]
print(f"{len(sents):,} sentences  ->  {len(train_sents):,} train / {len(test_sents):,} test")
print("Example:", train_sents[0][:12])

In [ ]:
# ---- The vocabulary with <unk>, <s>, </s> ----------------------------------
# Words that occur only ONCE in training become <unk> -> the model learns a
# distribution for unknown words, and the perplexity stays finite.
BOS, EOS, UNK = "<s>", "</s>", "<unk>"

train_counts = Counter(w for s in train_sents for w in s)
vocab = {w for w, c in train_counts.items() if c >= 2}
vocab |= {BOS, EOS, UNK}
print(f"Vocabulary size |V| = {len(vocab):,}")

def normalize(sent):
    return [BOS] + [w if w in vocab else UNK for w in sent] + [EOS]

train = [normalize(s) for s in train_sents]
test = [normalize(s) for s in test_sents]

In [ ]:
# ---- N-gram counts (unigram, bigram, trigram) ------------------------------
uni = Counter()
bi = defaultdict(Counter)     # bi[w1][w2] = C(w1,w2)
tri = defaultdict(Counter)    # tri[(w1,w2)][w3] = C(w1,w2,w3)

for s in train:
    for i, w in enumerate(s):
        uni[w] += 1
        if i >= 1:
            bi[s[i-1]][w] += 1
        if i >= 2:
            tri[(s[i-2], s[i-1])][w] += 1

N_uni = sum(uni.values())
V = len(vocab)
print(f"Tokens in total N = {N_uni:,},  |V| = {V:,}")
print("most frequent words:", uni.most_common(8))

### Task 1 — Add-k probabilities and perplexity
Implement the smoothed probabilities (1a) and the perplexity (1b). Compare the
uni-/bi-/trigram.

In [ ]:
# ---- TASK 1a: add-k smoothed probabilities --------------------------------
# The add-k formula (script 1.3):  P(w | context) = (C(context,w) + k) / (C(context) + k*|V|)
# Use uni, bi, tri and N_uni, V.
def p_unigram(w, k=1.0):
    # TODO
    raise NotImplementedError

def p_bigram(w, w1, k=1.0):
    # TODO:  (bi[w1][w] + k) / (uni[w1] + k*V)
    raise NotImplementedError

def p_trigram(w, w1, w2, k=1.0):
    # TODO:  the context is (w2, w1);  the denominator = sum(tri[(w2,w1)].values()) + k*V
    raise NotImplementedError

print("P(holmes | mr) =", round(p_bigram("holmes", "mr"), 5))

In [ ]:
# ---- TASK 1b: perplexity ---------------------------------------------------
# PP = exp( -1/N * sum_i log P(w_i | context) ),  summed over all positions i>=1
# of all test sentences;  N = the number of those positions.
def perplexity(sentences, prob_fn):
    # prob_fn(sent, i) returns P(w_i | the preceding words)
    # TODO
    raise NotImplementedError

pp_uni = perplexity(test, lambda s, i: p_unigram(s[i]))
pp_bi  = perplexity(test, lambda s, i: p_bigram(s[i], s[i-1]))
pp_tri = perplexity(test, lambda s, i: p_trigram(s[i], s[i-1], s[i-2]) if i >= 2
                                        else p_bigram(s[i], s[i-1]))
print(f"Perplexity (add-1):  unigram {pp_uni:7.1f} | bigram {pp_bi:7.1f} | trigram {pp_tri:7.1f}")

### Task 2 — Interpolation
Mix the three orders. With a small $k$ the perplexity should fall clearly below
that of the individual models.

In [ ]:
# ---- TASK 2: interpolation -------------------------------------------------
# P_interp = l1*P_uni + l2*P_bi + l3*P_tri  (with l1+l2+l3 = 1).
# Use a SMALLER k (e.g. 0.01) — add-1 is too crude for a large |V|.
# At the start of a sentence (i<2) there is no trigram context: use the bigram there.
def p_interp(s, i, l1=0.1, l2=0.3, l3=0.6, k=0.01):
    # TODO
    raise NotImplementedError

pp_interp = perplexity(test, p_interp)
print(f"Perplexity (interpolation): {pp_interp:7.1f}")

### Task 3 — Text generation
Sample sentences from the bigram model. The text sounds "Holmes-like" but is
grammatically crude — exactly the limit of n-grams.

In [ ]:
# ---- TASK 3: text generation by sampling -----------------------------------
# Start at <s>. Repeatedly draw the next word from the bigram model
# bi[last_word] (words weighted by their counts — random.choices).
# Stop at </s> or at max_len. Return the sentence without <s>/</s>.
def generate(max_len=25, seed=0):
    rng = random.Random(seed)
    # TODO
    raise NotImplementedError

for seed in range(5):
    print(" *", generate(seed=seed))

## Reflection (briefly, in writing)
1. A surprise: with **add-1** the trigram is *worse* than the unigram. Why?
   (Over-smoothing: with a large $|V|$, $k\cdot|V|$ in the denominator eats the
   whole mass.) And why does interpolation with a small $k$ fix that?
2. What happens to the perplexity if you use an unsmoothed model in `perplexity`
   and a test word never occurred in that context during training?
3. Why does interpolation improve on the pure trigram? (Script: falling back to
   the lower order when the data are thin.)
4. The generated text is locally plausible but globally nonsense. Which property
   of language can n-grams not capture in principle? (An outlook on module 09.)

Reference answers are at the end of the solution in the folder `solution/`.